<a href="https://colab.research.google.com/github/miray7yuce/quadcopter-rl-copilot/blob/main/notebooks/quadcopter_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q stable-baselines3 gymnasium
!pip install -q jsbsim==1.2.4
!pip install -q pyyaml
!pip install -q optuna

import jsbsim
print(jsbsim.__version__)

1.2.4


In [2]:
!pip install fastapi uvicorn websockets --quiet


In [2]:
#repodaki güncel dosyaları çeker PUSHLAMADAN ÇALIŞTIRMA
from google.colab import userdata
import os

USER  = "miray7yuce"
REPO  = "quadcopter-rl-copilot"
TOKEN = userdata.get('GH_TOKEN')

!git config --global user.email "miray7yuce@gmail.com"
!git config --global user.name "miray7yuce"

os.environ['REMOTE'] = f"https://{TOKEN}@github.com/{USER}/{REPO}.git"
!rm -rf /content/repo
!git clone -q $REMOTE /content/repo
!ls -a /content/repo

.			 notebooks		    requirements.txt
..			 ppo_final.acmi		    runs
configs			 ppo_flight_final.acmi	    sac_final.acmi
droneSim_realtime.html	 ppo_flight_telemetry.csv   src
f450-drone-framestl.stl  ppo_telemetry.csv	    telemetry_for_html.json
.git			 ppo_vs_sac_comparison.png
.gitignore		 README.md


In [3]:
%%writefile /content/repo/src/drone_rl/realtime_server.py
"""F450 flight - gercek zamanli (WASD + ok tuslariyla oynanabilir) backend.

Colab icinde calisir: FastAPI + WebSocket ile her karede (frame) ortami
bir adim ilerletir, ya PPO'nun urettigi aksiyonu ya da WASD/ok
tuslarindan gelen aksiyonu kullanir, sonucu tarayiciya gonderir.

Kontroller (tarayicida):
  W / S       -> ileri / geri (pitch)
  A / D       -> sola / saga (roll)
  Yukari Ok   -> yukselme (throttle+)
  Asagi Ok    -> alcalma (throttle-)
  Hicbir tus basili degilse -> PPO otomatik ucusa devam eder

Retraining GEREKMIYOR - ayni egitilmis model (model_final.zip +
vecnormalize.pkl) burada da kullaniliyor, sadece nerede/nasil
calistirdigimiz degisiyor.
"""

import asyncio
import json
from pathlib import Path

import numpy as np
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import FileResponse
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize

from drone_rl.config import load_config
from drone_rl.env_factory import make_flight_env, make_flight_eval_vec_env
from drone_rl.evaluate import resolve_model_paths


# --- Manuel kontrol hissi icin ayarlanabilir sabitler ---
# Bunlar fiziksel dogruluk degil, "oyun hissi" sabitleri - istedigin gibi
# degistirebilirsin. Onceki degerlere (0.30/0.30/0.40) gore buyutuldu,
# cunku kucuk degerler quadin gercek kutlesi/ataleti yuzunden cok yavas
# hizlanip yavasliyordu - daha buyuk degerler daha "gercek zamanli" hissettirir.
PITCH_MAG = 0.55   # W/S -> ileri/geri
ROLL_MAG = 0.55    # A/D -> sola/saga
ALT_MAG = 0.65     # E/F -> yukselme/alcalma

# Cevirirken (W/A/S/D) irtifayi sabit tutan geri besleme (feedback)
# katsayisi. Drone yana/one yatinca toplam itki artik tam dikey olmadigi
# icin dogal olarak alcalir (gercek quadcopterlarda da boyledir) - bu
# katsayi o kaybi telafi ediyor.
ALT_HOLD_KP = 0.08
ALT_HOLD_MAX = 0.5


class ManualController:
    """W/A/S/D = ileri/geri/sola/saga (pitch/roll ile), E/F = yukselme/alcalma.

    W/A/S/D basiliyken E/F basili DEGILSE, egilme (tilt) kaynakli dogal
    irtifa kaybini otomatik telafi ederek irtifayi kilitli tutar - yoksa
    "A/D irtifayi da degistiriyor" gibi kafa karistirici bir yan etki
    olur (bu FIZIKSEL bir etki, motor mixing hatasi degil: drone
    yatinca toplam itkinin dikey bilesimi azalir).

    E veya F basilinca kilit birakilir, dogrudan tam guclu yukselme/
    alcalma komutu verilir. Tum tuslar birakilinca (PPO otomatik pilota
    donulunce) kilit sifirlanir - manuel kontrole tekrar girildiginde
    O ANKI irtifadan yeniden kilitlenir, eski/bayat bir degerden degil.
    """

    def __init__(self):
        self.alt_lock = None

    def compute_action(self, keys: dict, current_alt_ft: float) -> np.ndarray:
        pitch = PITCH_MAG if keys.get("w") else (-PITCH_MAG if keys.get("s") else 0.0)
        roll = ROLL_MAG if keys.get("d") else (-ROLL_MAG if keys.get("a") else 0.0)

        if keys.get("e") or keys.get("f"):
            self.alt_lock = None
            throttle = ALT_MAG if keys.get("e") else -ALT_MAG
        else:
            if self.alt_lock is None:
                self.alt_lock = current_alt_ft
            throttle = float(np.clip(
                ALT_HOLD_KP * (self.alt_lock - current_alt_ft), -ALT_HOLD_MAX, ALT_HOLD_MAX
            ))

        motor_fl = throttle + pitch + roll
        motor_fr = throttle + pitch - roll
        motor_rl = throttle - pitch + roll
        motor_rr = throttle - pitch - roll

        action = np.array([motor_fl, motor_fr, motor_rl, motor_rr], dtype=np.float32)
        return np.clip(action, -1.0, 1.0)

    def reset_lock(self):
        self.alt_lock = None


def any_key_pressed(keys: dict) -> bool:
    # DIKKAT: ok tuslari artik burada YOK - onlar sadece kamera icin,
    # drone kontrolune dahil degiller (E/F irtifa icin kullaniliyor).
    return any(keys.get(k) for k in ("w", "a", "s", "d", "e", "f"))


class NormalizerStats:
    """VecNormalize'in mean/var degerlerini tasiyan hafif bir tasiyici.

    Egitimde kullanilan normalize etme formulunu (obs -> normalized obs),
    tek bir canli gozlem uzerinde MANUEL olarak uygulayabilmek icin -
    canli dongude gercek bir VecEnv/VecNormalize wrapper'i kullanmiyoruz
    (tek ortam, tek adim, sürekli acik kalan bir dongu oldugu icin daha
    basit). Bu formul VecNormalize.normalize_obs() ile birebir ayni.
    """

    def __init__(self, vecnorm: VecNormalize):
        self.mean = vecnorm.obs_rms.mean.astype(np.float32)
        self.var = vecnorm.obs_rms.var.astype(np.float32)
        self.epsilon = vecnorm.epsilon
        self.clip_obs = vecnorm.clip_obs

    def normalize(self, obs: np.ndarray) -> np.ndarray:
        normed = (obs - self.mean) / np.sqrt(self.var + self.epsilon)
        return np.clip(normed, -self.clip_obs, self.clip_obs).astype(np.float32)


def load_policy(run: str, config: str, use_best: bool = False):
    cfg = load_config(config)
    run_path = Path(run)
    model_path, vecnorm_path = resolve_model_paths(run_path, use_best)

    if not vecnorm_path.exists():
        raise FileNotFoundError(f"VecNormalize dosyasi bulunamadi: {vecnorm_path}")

    # Gercek bir egitim/eval dongusu kurmuyoruz - sadece VecNormalize'in
    # ogrendigi mean/var istatistiklerini disari cikarmak icin gecici
    # (dummy) bir vec-env uzerinden yukluyoruz.
    dummy_venv = make_flight_eval_vec_env(cfg.flight_env)
    vecnorm = VecNormalize.load(str(vecnorm_path), dummy_venv)
    stats = NormalizerStats(vecnorm)

    model = PPO.load(str(model_path), device="cpu")
    return model, stats, cfg


app = FastAPI()

# start_server() cagrildiginda doldurulur; /ws endpoint'i buradan okur.
STATE = {"model": None, "stats": None, "cfg": None, "html_path": None}


@app.get("/")
def index():
    return FileResponse(STATE["html_path"])


# Basariyla hedefe ulasildiktan sonra, hemen resetlemeden once ekranda
# ne kadar sure daha (saniye) izleyelim - saf gorsellestirme amacli,
# env'in kendi terminated=True mantigini (egitimde kullanilan) DEGISTIRMIYORUZ,
# sadece sunucu tarafinda "reset()'i cagirmayi geciktiriyoruz".
SUCCESS_LINGER_SECONDS = 3.0


@app.websocket("/ws")
async def flight_loop(websocket: WebSocket):
    await websocket.accept()

    model = STATE["model"]
    stats = STATE["stats"]
    cfg = STATE["cfg"]

    env = make_flight_env(cfg.flight_env)
    obs, _ = env.reset()

    # None: normal ucus. Sayi: "basariyla ulasti, su kadar adim sonra
    # resetlenecek" geri sayimi. Bu sayede terminated=True dondugu anda
    # DEGIL, SUCCESS_LINGER_SECONDS kadar sonra reset() cagriliyor -
    # boylece "TARGET REACHED" durumunu ekranda birkac saniye gorebiliyoruz.
    linger_steps_total = max(1, int(SUCCESS_LINGER_SECONDS / env.control_dt))
    linger_remaining = None
    manual_ctrl = ManualController()

    # WASD durumu ayri bir "receiver" task'inde tutuluyor, boylece ana
    # fizik dongusu tus mesaji beklemek zorunda kalmadan kendi hizinda
    # (control_dt) akmaya devam edebiliyor. Poll+timeout yontemi yerine
    # bu, hem daha az CPU harcar hem tus olaylarini kacirmaz.
    current_keys = {}

    async def receiver():
        nonlocal current_keys
        try:
            while True:
                raw = await websocket.receive_text()
                current_keys = json.loads(raw) if raw else {}
        except WebSocketDisconnect:
            pass

    receiver_task = asyncio.create_task(receiver())

    try:
        while True:
            keys = current_keys

            if any_key_pressed(keys):
                current_alt_ft = env.fdm["position/h-agl-ft"]
                action = manual_ctrl.compute_action(keys, current_alt_ft)
                mode = "manual"
            else:
                manual_ctrl.reset_lock()
                norm_obs = stats.normalize(obs).reshape(1, -1)
                action, _ = model.predict(norm_obs, deterministic=True)
                action = action[0]
                mode = "auto"

            obs, reward, terminated, truncated, info = env.step(action)

            episode_reset = False

            if linger_remaining is not None:
                # Basari sonrasi "bekleme" penceresindeyiz. Cakisma olursa
                # beklemeden hemen resetle; yoksa geri sayimi azalt.
                linger_remaining -= 1
                if info.get("crashed") or linger_remaining <= 0:
                    obs, _ = env.reset()
                    episode_reset = True
                    linger_remaining = None
                # yoksa: reset ETME, env'in terminated=True demesine ragmen
                # step() atmaya devam ediyoruz - JSBSim'in fizigi bunu
                # umursamiyor, sadece bizim reset() cagirip cagirmamamiz
                # onemli.
            elif terminated or truncated:
                if info.get("reached_target") and not info.get("crashed"):
                    # Basariyla ulasti - hemen resetlemek yerine bekleme
                    # geri sayimini baslat.
                    linger_remaining = linger_steps_total
                else:
                    obs, _ = env.reset()
                    episode_reset = True

            payload = dict(info)
            payload["mode"] = mode
            payload["episode_reset"] = episode_reset
            payload["target_altitude_ft"] = env.target_altitude
            payload["control_dt"] = env.control_dt
            payload["motor_throttle"] = np.clip(
                env.hover_throttle + action * env.throttle_range, 0.0, 1.0
            ).tolist()

            await websocket.send_text(json.dumps(payload))
            await asyncio.sleep(env.control_dt)

    except WebSocketDisconnect:
        pass
    finally:
        receiver_task.cancel()


def start_server(run: str, config: str, html_path: str, use_best: bool = False, port: int = 8000):
    """Colab hucresinden cagrilacak baslatma fonksiyonu.

    Ornek kullanim (Colab hucresi):
        from drone_rl.realtime_server import start_server
        start_server(
            run="/content/runs/flight_ppo_v4",
            config="/content/repo/configs/ppo_flight.yaml",
            html_path="/content/droneSim_realtime.html",
        )
        from google.colab.output import serve_kernel_port_as_window
        serve_kernel_port_as_window(8000)
    """
    import threading
    import uvicorn

    model, stats, cfg = load_policy(run, config, use_best)
    STATE["model"] = model
    STATE["stats"] = stats
    STATE["cfg"] = cfg
    STATE["html_path"] = html_path

    thread = threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=port, log_level="warning"),
        daemon=True,
    )
    thread.start()
    print(f"Sunucu baslatildi (arka planda, port {port}).")
    print("Simdi asagidaki hucreyi calistirip acilan pencereye/linke tikla.")


Overwriting /content/repo/src/drone_rl/realtime_server.py


In [24]:
%%writefile /content/repo/droneSim_realtime.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8" />
<title>F450 Flight — Real-Time Simulator</title>
<meta name="viewport" content="width=device-width, initial-scale=1" />
<style>
  :root{
    --bg-0:#0b0f14; --bg-1:#111820; --bg-2:#161f29; --line:#26323e;
    --ink-0:#e8eef3; --ink-1:#9fb0bd; --ink-2:#5f7180;
    --accent:#4fd1c5; --accent-dim:#2a6b66; --warn:#e0a05a; --danger:#e0596a;
    --auto:#4fd1c5; --manual:#e0a05a;
    --mono: "JetBrains Mono","SF Mono",Consolas,monospace;
    --sans: -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;
  }
  *{box-sizing:border-box;}
  html,body{ margin:0; height:100%; background:var(--bg-0); color:var(--ink-0); font-family:var(--sans); }
  #app{ display:flex; flex-direction:column; height:100vh; }

  header{
    display:flex; align-items:center; gap:18px; padding:12px 18px;
    background:var(--bg-1); border-bottom:1px solid var(--line); flex-wrap:wrap;
  }
  header h1{ font-size:15px; font-weight:600; margin:0; white-space:nowrap; }
  header h1 span{ color:var(--accent); }

  .picker{ display:flex; align-items:center; gap:8px; }
  .picker label{
    font-size:12px; color:var(--ink-1); cursor:pointer;
    border:1px solid var(--line); background:var(--bg-2);
    padding:7px 12px; border-radius:6px; transition:border-color .15s;
  }
  .picker label:hover{ border-color:var(--accent-dim); }
  .picker input[type=file]{ display:none; }
  .picker .status{ font-size:11px; font-family:var(--mono); color:var(--ink-2); min-width:110px; }
  .picker .status.ok{ color:var(--accent); }

  .conn-status{
    margin-left:auto; display:flex; align-items:center; gap:8px;
    font-family:var(--mono); font-size:12px;
  }
  .conn-dot{ width:8px; height:8px; border-radius:50%; background:var(--ink-2); }
  .conn-dot.connected{ background:var(--accent); }
  .conn-dot.disconnected{ background:var(--danger); }

  .mode-badge{
    font-family:var(--mono); font-size:11px; font-weight:700;
    padding:4px 10px; border-radius:12px; letter-spacing:.4px;
  }
  .mode-badge.auto{ background:rgba(79,209,197,.15); color:var(--auto); border:1px solid var(--auto); }
  .mode-badge.manual{ background:rgba(224,160,90,.15); color:var(--manual); border:1px solid var(--manual); }

  main{ flex:1; display:grid; grid-template-columns: 1fr 320px; min-height:0; }

  #viewport-wrap{ position:relative; background:radial-gradient(ellipse at 50% 30%, #131c26, #0b0f14 70%); }
  #three-canvas{ width:100%; height:100%; display:block; }

  #hud{
    position:absolute; top:14px; left:14px;
    background:rgba(17,24,32,.82); border:1px solid var(--line);
    border-radius:8px; padding:10px 14px;
    font-family:var(--mono); font-size:12px; line-height:1.65;
    color:var(--ink-0); pointer-events:none; backdrop-filter: blur(4px);
    min-width:190px;
  }
  #hud .row{ display:flex; justify-content:space-between; gap:18px; }
  #hud .row .k{ color:var(--ink-2); }
  #hud .row .v{ font-weight:600; }
  #hud .crash{ color:var(--danger); }
  #hud .success{ color:var(--accent); }

  #controls-hint{
    position:absolute; bottom:14px; left:14px;
    background:rgba(17,24,32,.82); border:1px solid var(--line);
    border-radius:8px; padding:10px 14px; font-size:11px; color:var(--ink-1);
    font-family:var(--mono); line-height:1.7;
  }
  #controls-hint b{ color:var(--ink-0); }
  .key-active{ color:var(--manual) !important; }

  aside#charts{
    background:var(--bg-1); border-left:1px solid var(--line);
    padding:14px; display:flex; flex-direction:column; gap:14px; overflow:auto;
  }
  .chart-card{ background:var(--bg-2); border:1px solid var(--line); border-radius:8px; padding:10px 10px 6px; }
  .chart-card .title{ font-size:11px; color:var(--ink-1); margin-bottom:6px; }
  .chart-card canvas{ width:100%; height:110px; display:block; }
  .legend{ display:flex; gap:12px; margin-top:4px; font-size:10px; color:var(--ink-2); }
  .legend .dot{ display:inline-block; width:8px; height:8px; border-radius:50%; margin-right:4px; }

  .empty-note{ color:var(--ink-2); font-size:12px; padding:18px 6px; line-height:1.6; }
</style>
</head>
<body>
<div id="app">
  <header>
    <h1>F450 <span>Real-Time</span> Flight</h1>

    <div class="picker">
      <label for="stl-input">Select F450 STL Model</label>
      <input id="stl-input" type="file" accept=".stl" />
      <span id="stl-status" class="status">not loaded</span>
    </div>

    <span id="mode-badge" class="mode-badge auto">AUTO (PPO)</span>

    <div class="conn-status">
      <span class="conn-dot" id="conn-dot"></span>
      <span id="conn-text">connecting…</span>
    </div>
  </header>

  <main>
    <section id="viewport-wrap">
      <canvas id="three-canvas"></canvas>

      <div id="hud">
        <div class="row"><span class="k">t</span><span class="v" id="hud-t">–</span></div>
        <div class="row"><span class="k">altitude</span><span class="v" id="hud-alt">–</span></div>
        <div class="row"><span class="k">target alt.</span><span class="v" id="hud-target">–</span></div>
        <div class="row"><span class="k">alt. error</span><span class="v" id="hud-alterr">–</span></div>
        <div class="row"><span class="k">roll</span><span class="v" id="hud-roll">–</span></div>
        <div class="row"><span class="k">pitch</span><span class="v" id="hud-pitch">–</span></div>
        <div class="row"><span class="k">yaw</span><span class="v" id="hud-yaw">–</span></div>
        <div class="row"><span class="k">status</span><span class="v" id="hud-crash">–</span></div>
        <div class="row"><span class="k">motors</span><span class="v" id="hud-motors">–</span></div>
      </div>

      <div id="controls-hint">
        <b>Controls</b><br/>
        <span id="key-w">W</span>/<span id="key-s">S</span> forward/back &nbsp;
        <span id="key-a">A</span>/<span id="key-d">D</span> left/right<br/>
        <span id="key-e">E</span>/<span id="key-f">F</span> climb/descend<br/>
        arrow keys pan the camera view (no effect on the drone)<br/>
        release all keys → PPO autopilot resumes
      </div>
    </section>

    <aside id="charts">
      <div class="chart-card">
        <div class="title">Altitude Error (ft)</div>
        <canvas id="chart-alt"></canvas>
      </div>
      <div class="chart-card">
        <div class="title">Roll / Pitch (degrees)</div>
        <canvas id="chart-tilt"></canvas>
        <div class="legend">
          <span><span class="dot" style="background:#4fd1c5"></span>Roll</span>
          <span><span class="dot" style="background:#e0a05a"></span>Pitch</span>
        </div>
      </div>
      <div class="empty-note" id="empty-note">
        Select an STL model above. The flight starts automatically once it loads — PPO flies until you press a control key.
      </div>
    </aside>
  </main>
</div>

<script type="importmap">
{
  "imports": {
    "three": "https://unpkg.com/three@0.160.0/build/three.module.js",
    "three/addons/": "https://unpkg.com/three@0.160.0/examples/jsm/"
  }
}
</script>

<script type="module">
import * as THREE from "three";
import { OrbitControls } from "three/addons/controls/OrbitControls.js";
import { STLLoader } from "three/addons/loaders/STLLoader.js";

/* ------------------------------------------------------------------ */
/* Constants                                                            */
/* ------------------------------------------------------------------ */
const FT_PER_M = 3.28084;
const DESIRED_SPAN_FT = 4.0;
const TRAIL_LEN = 150;
const CHART_WINDOW = 200; // canli grafiklerde tutulan son ornek sayisi

/* ------------------------------------------------------------------ */
/* DOM references                                                       */
/* ------------------------------------------------------------------ */
const canvas = document.getElementById("three-canvas");
const stlInput = document.getElementById("stl-input");
const stlStatus = document.getElementById("stl-status");
const emptyNote = document.getElementById("empty-note");
const modeBadge = document.getElementById("mode-badge");
const connDot = document.getElementById("conn-dot");
const connText = document.getElementById("conn-text");

const hud = {
  t: document.getElementById("hud-t"),
  alt: document.getElementById("hud-alt"),
  target: document.getElementById("hud-target"),
  alterr: document.getElementById("hud-alterr"),
  roll: document.getElementById("hud-roll"),
  pitch: document.getElementById("hud-pitch"),
  yaw: document.getElementById("hud-yaw"),
  crash: document.getElementById("hud-crash"),
  motors: document.getElementById("hud-motors"),
};

const chartCanvases = {
  alt: document.getElementById("chart-alt"),
  tilt: document.getElementById("chart-tilt"),
};

/* ------------------------------------------------------------------ */
/* Three.js scene setup (identical structure to the offline viewer)     */
/* ------------------------------------------------------------------ */
const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(45, 1, 0.01, 5000);
camera.up.set(0, 0, 1);

const renderer = new THREE.WebGLRenderer({ canvas, antialias: true });
renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));

const controls = new OrbitControls(camera, renderer.domElement);
controls.enableDamping = true;
controls.dampingFactor = 0.08;
controls.listenToKeyEvents(window);
controls.keyPanSpeed = 20.0;
controls.minDistance = 4;
controls.maxDistance = 120;
controls.zoomSpeed = 1.25;

scene.add(new THREE.AmbientLight(0xffffff, 0.55));
const sun = new THREE.DirectionalLight(0xffffff, 0.9);
sun.position.set(40, -30, 60);
scene.add(sun);
const fill = new THREE.DirectionalLight(0x88aacc, 0.35);
fill.position.set(-30, 40, 20);
scene.add(fill);

let groundGrid = new THREE.GridHelper(40, 20, 0x2a3644, 0x1a232c);
groundGrid.rotation.x = Math.PI / 2;
scene.add(groundGrid);

const targetPlaneMat = new THREE.MeshBasicMaterial({
  color: 0x4fd1c5, transparent: true, opacity: 0.10, side: THREE.DoubleSide,
});
let targetPlane = new THREE.Mesh(new THREE.PlaneGeometry(40, 40), targetPlaneMat);
scene.add(targetPlane);

const droneGroup = new THREE.Group();
scene.add(droneGroup);
let modelMesh = null;

const shadowMarker = new THREE.Mesh(
  new THREE.CircleGeometry(0.5, 24),
  new THREE.MeshBasicMaterial({ color: 0xe0a05a, transparent: true, opacity: 0.85 })
);
scene.add(shadowMarker);

const vertLineGeom = new THREE.BufferGeometry().setFromPoints([
  new THREE.Vector3(0, 0, 0), new THREE.Vector3(0, 0, 0),
]);
const vertLineMat = new THREE.LineDashedMaterial({ color: 0xe0a05a, dashSize: 0.6, gapSize: 0.4, transparent: true, opacity: 0.7 });
const vertLine = new THREE.Line(vertLineGeom, vertLineMat);
scene.add(vertLine);

let trailPoints = [];
const trailMat = new THREE.LineBasicMaterial({ color: 0x7f93a3, transparent: true, opacity: 0.55 });
let trailLine = new THREE.Line(new THREE.BufferGeometry(), trailMat);
scene.add(trailLine);

let cameraInitialized = false;

function resizeRenderer() {
  const w = canvas.clientWidth, h = canvas.clientHeight;
  renderer.setSize(w, h, false);
  camera.aspect = w / Math.max(h, 1);
  camera.updateProjectionMatrix();
}
window.addEventListener("resize", resizeRenderer);

/* ------------------------------------------------------------------ */
/* STL loading                                                          */
/* ------------------------------------------------------------------ */
const stlLoader = new STLLoader();

stlInput.addEventListener("change", (ev) => {
  const file = ev.target.files[0];
  if (!file) return;
  const reader = new FileReader();
  reader.onload = (e) => {
    const geometry = stlLoader.parse(e.target.result);
    geometry.computeBoundingBox();
    geometry.center();
    geometry.computeVertexNormals();

    const size = new THREE.Vector3();
    geometry.boundingBox.getSize(size);
    const maxDim = Math.max(size.x, size.y, size.z) || 1;
    const scaleFactor = DESIRED_SPAN_FT / maxDim;

    const material = new THREE.MeshStandardMaterial({ color: 0x9fb4c7, metalness: 0.25, roughness: 0.55 });

    if (modelMesh) droneGroup.remove(modelMesh);
    modelMesh = new THREE.Mesh(geometry, material);
    modelMesh.scale.setScalar(scaleFactor);
    droneGroup.add(modelMesh);

    stlStatus.textContent = "loaded ✓";
    stlStatus.classList.add("ok");
    emptyNote.style.display = "none";
  };
  reader.readAsArrayBuffer(file);
});

/* ------------------------------------------------------------------ */
/* WebSocket connection - live frame stream                             */
/* ------------------------------------------------------------------ */
let visualScale = 1.0;
let latestFrame = null;

// Kayan pencereli (rolling window) grafik verisi - tum episode'u degil,
// sadece son CHART_WINDOW ornegi tutuyoruz.
const chartBuf = { t: [], altErr: [], rollDeg: [], pitchDeg: [] };

function pushChartSample(f) {
  chartBuf.t.push(f.t_local);
  chartBuf.altErr.push(f.alt_err_ft);
  chartBuf.rollDeg.push(f.roll_rad * 180 / Math.PI);
  chartBuf.pitchDeg.push(f.pitch_rad * 180 / Math.PI);
  if (chartBuf.t.length > CHART_WINDOW) {
    chartBuf.t.shift(); chartBuf.altErr.shift();
    chartBuf.rollDeg.shift(); chartBuf.pitchDeg.shift();
  }
}

let localClock = 0;

function connect() {
  const wsProtocol = window.location.protocol === "https:" ? "wss:" : "ws:";
  const ws = new WebSocket(`${wsProtocol}//${window.location.host}/ws`);

  ws.onopen = () => {
    connDot.className = "conn-dot connected";
    connText.textContent = "connected";
  };
  ws.onclose = () => {
    connDot.className = "conn-dot disconnected";
    connText.textContent = "disconnected — retrying…";
    setTimeout(connect, 1500);
  };
  ws.onerror = () => { ws.close(); };

  ws.onmessage = (msg) => {
    const f = JSON.parse(msg.data);
    if (f.episode_reset) {
      trailPoints = [];
      chartBuf.t = []; chartBuf.altErr = []; chartBuf.rollDeg = []; chartBuf.pitchDeg = [];
      localClock = 0;
    }
    localClock += f.control_dt || 0.05;
    f.t_local = localClock;
    latestFrame = f;
  };

  window._sendKeys = (keys) => {
    if (ws.readyState === WebSocket.OPEN) ws.send(JSON.stringify(keys));
  };
}
connect();

/* ------------------------------------------------------------------ */
/* Keyboard input - WASD + arrows                                       */
/* ------------------------------------------------------------------ */
const TRACKED_KEYS = ["w", "a", "s", "d", "e", "f"];
const keyState = { w: false, a: false, s: false, d: false, e: false, f: false };
const keyHintEls = {
  w: document.getElementById("key-w"), a: document.getElementById("key-a"),
  s: document.getElementById("key-s"), d: document.getElementById("key-d"),
  e: document.getElementById("key-e"), f: document.getElementById("key-f"),
};

function normalizedKey(e) {
  // DIKKAT: ok tuslari artik burada YOK - onlar drone'u degil, sadece
  // OrbitControls uzerinden kamerayi kontrol ediyor (kendi ic listener'iyla).
  const lower = e.key.toLowerCase();
  return TRACKED_KEYS.includes(lower) ? lower : null;
}

window.addEventListener("keydown", (e) => {
  const k = normalizedKey(e);
  if (!k) return;
  if (!keyState[k]) {
    keyState[k] = true;
    keyHintEls[k]?.classList.add("key-active");
    window._sendKeys && window._sendKeys(keyState);
  }
});
window.addEventListener("keyup", (e) => {
  const k = normalizedKey(e);
  if (!k) return;
  if (keyState[k]) {
    keyState[k] = false;
    keyHintEls[k]?.classList.remove("key-active");
    window._sendKeys && window._sendKeys(keyState);
  }
});

/* ------------------------------------------------------------------ */
/* Rendering a live frame                                               */
/* ------------------------------------------------------------------ */
function ensureCameraFramed(f) {
  if (cameraInitialized) return;
  cameraInitialized = true;

  const gridSize = 40;
  scene.remove(targetPlane);
  targetPlane = new THREE.Mesh(new THREE.PlaneGeometry(gridSize, gridSize), targetPlaneMat);
  targetPlane.position.z = f.target_altitude_ft;
  scene.add(targetPlane);

  const dist = 30;
  const elev = THREE.MathUtils.degToRad(35);
  const azim = THREE.MathUtils.degToRad(45);
  camera.position.set(dist * Math.cos(elev) * Math.cos(azim), dist * Math.cos(elev) * Math.sin(azim), f.target_altitude_ft + dist * Math.sin(elev) * 0.4);
  controls.target.set(0, 0, f.target_altitude_ft * 0.5);
  controls.update();
}

function renderFrame(f) {
  ensureCameraFramed(f);

  const vx = f.x_m * FT_PER_M;
  const vy = f.y_m * FT_PER_M;
  const vz = f.alt_ft;

  droneGroup.position.set(vx, vy, vz);
  droneGroup.rotation.set(f.roll_rad, f.pitch_rad, f.yaw_rad, "XYZ");

  shadowMarker.position.set(vx, vy, 0.02);
  vertLineGeom.setFromPoints([new THREE.Vector3(vx, vy, 0), new THREE.Vector3(vx, vy, vz)]);
  vertLine.computeLineDistances();

  trailPoints.push(new THREE.Vector3(vx, vy, vz));
  if (trailPoints.length > TRAIL_LEN) trailPoints.shift();
  trailLine.geometry.dispose();
  trailLine.geometry = new THREE.BufferGeometry().setFromPoints(trailPoints);

  hud.t.textContent = `${f.t_local.toFixed(1)} s`;
  hud.alt.textContent = `${f.alt_ft.toFixed(2)} ft`;
  hud.target.textContent = `${f.target_altitude_ft.toFixed(1)} ft`;
  hud.alterr.textContent = `${f.alt_err_ft.toFixed(2)} ft`;
  hud.roll.textContent = `${(f.roll_rad * 180 / Math.PI).toFixed(2)}°`;
  hud.pitch.textContent = `${(f.pitch_rad * 180 / Math.PI).toFixed(2)}°`;
  hud.yaw.textContent = `${(f.yaw_rad * 180 / Math.PI).toFixed(2)}°`;

  if (f.crashed) {
    hud.crash.textContent = "CRASHED"; hud.crash.className = "v crash";
  } else if (f.reached_target) {
    hud.crash.textContent = "TARGET REACHED ✓"; hud.crash.className = "v success";
  } else {
    hud.crash.textContent = "normal"; hud.crash.className = "v";
  }

  if (f.motor_throttle) {
    hud.motors.textContent = f.motor_throttle.map(v => `${Math.round(v * 100)}%`).join(" / ");
  }

  modeBadge.textContent = f.mode === "manual" ? "MANUAL" : "AUTO (PPO)";
  modeBadge.className = `mode-badge ${f.mode === "manual" ? "manual" : "auto"}`;

  pushChartSample(f);
  drawCharts();
}

/* ------------------------------------------------------------------ */
/* Rolling-window charts (simple canvas, no external library)          */
/* ------------------------------------------------------------------ */
function fitCanvas(cv) {
  const rect = cv.getBoundingClientRect();
  cv.width = Math.max(1, Math.floor(rect.width * devicePixelRatio));
  cv.height = Math.max(1, Math.floor(rect.height * devicePixelRatio));
  const ctx = cv.getContext("2d");
  ctx.setTransform(devicePixelRatio, 0, 0, devicePixelRatio, 0, 0);
  return ctx;
}

function drawSeriesChart(cv, series, xArr) {
  if (xArr.length < 2) return;
  const ctx = fitCanvas(cv);
  const w = cv.clientWidth, h = cv.clientHeight;
  const padL = 34, padR = 8, padT = 8, padB = 16;
  ctx.clearRect(0, 0, w, h);

  let yMin = Infinity, yMax = -Infinity;
  series.forEach(s => s.values.forEach(v => { if (v < yMin) yMin = v; if (v > yMax) yMax = v; }));
  if (!isFinite(yMin)) { yMin = 0; yMax = 1; }
  if (yMax - yMin < 1e-6) { yMax += 1; yMin -= 1; }
  const yPad = (yMax - yMin) * 0.12;
  yMin -= yPad; yMax += yPad;

  const xMin = xArr[0], xMax = xArr[xArr.length - 1] || 1;
  const xToPx = (x) => padL + ((x - xMin) / (xMax - xMin || 1)) * (w - padL - padR);
  const yToPx = (v) => (h - padB) - ((v - yMin) / (yMax - yMin || 1)) * (h - padT - padB);

  ctx.strokeStyle = "#1f2933"; ctx.lineWidth = 1;
  for (let g = 0; g <= 3; g++) {
    const yy = padT + (g / 3) * (h - padT - padB);
    ctx.beginPath(); ctx.moveTo(padL, yy); ctx.lineTo(w - padR, yy); ctx.stroke();
  }

  ctx.font = "10px monospace"; ctx.fillStyle = "#5f7180"; ctx.textAlign = "right";
  ctx.fillText(yMax.toFixed(1), padL - 4, padT + 8);
  ctx.fillText(yMin.toFixed(1), padL - 4, h - padB);

  series.forEach(s => {
    ctx.beginPath(); ctx.strokeStyle = s.color; ctx.lineWidth = 1.5;
    xArr.forEach((x, i) => {
      const px = xToPx(x), py = yToPx(s.values[i]);
      if (i === 0) ctx.moveTo(px, py); else ctx.lineTo(px, py);
    });
    ctx.stroke();
  });
}

function drawCharts() {
  drawSeriesChart(chartCanvases.alt, [{ values: chartBuf.altErr, color: "#4fd1c5" }], chartBuf.t);
  drawSeriesChart(chartCanvases.tilt, [
    { values: chartBuf.rollDeg, color: "#4fd1c5" },
    { values: chartBuf.pitchDeg, color: "#e0a05a" },
  ], chartBuf.t);
}

/* ------------------------------------------------------------------ */
/* Main render loop - draws whatever the latest received frame is       */
/* ------------------------------------------------------------------ */
function animate() {
  requestAnimationFrame(animate);
  if (latestFrame) renderFrame(latestFrame);
  controls.update();
  resizeRenderer();
  renderer.render(scene, camera);
}
requestAnimationFrame(animate);
</script>
</body>
</html>


Overwriting /content/repo/droneSim_realtime.html


In [22]:
import sys
sys.path.insert(0, "/content/repo/src")

from drone_rl.realtime_server import start_server

start_server(
    run="/content/repo/runs/flight_ppo_v4",
    config="/content/repo/configs/ppo_flight.yaml",
    html_path="/content/repo/droneSim_realtime.html",
)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Sunucu baslatildi (arka planda, port 8000).
Simdi asagidaki hucreyi calistirip acilan pencereye/linke tikla.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [23]:

from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000)


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [5]:
import os
os.chdir('/content')
print(os.getcwd())

/content


In [6]:
!pip freeze | grep -iE "^(jsbsim|stable-baselines3|gymnasium|torch|numpy)=" > /content/repo/requirements.txt
!cat /content/repo/requirements.txt

gymnasium==1.3.0
jsbsim==1.2.4
numpy==2.1.3


In [7]:
import os, sys

BASE = "/content/repo"

for d in ["src/drone_rl/envs", "src/drone_rl/utils", "configs"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

for p in ["src/drone_rl", "src/drone_rl/envs", "src/drone_rl/utils"]:
    open(f"{BASE}/{p}/__init__.py", "a").close()

with open(f"{BASE}/.gitignore", "w") as f:
    f.write("__pycache__/\n*.zip\n*.pkl\nlogs/\nruns/\n.ipynb_checkpoints/\n")

sys.path.insert(0, f"{BASE}/src")

!find /content/repo -not -path '*/.git/*' -type f | sort

os.environ['PYTHONPATH'] = f"{BASE}/src"

/content/repo/configs/ppo_flight.yaml
/content/repo/configs/ppo_hover_customnet.yaml
/content/repo/configs/ppo_hover.yaml
/content/repo/droneSim_realtime.html
/content/repo/f450-drone-framestl.stl
/content/repo/.gitignore
/content/repo/notebooks/quadcopter_rl.ipynb
/content/repo/ppo_final.acmi
/content/repo/ppo_flight_final.acmi
/content/repo/ppo_flight_telemetry.csv
/content/repo/ppo_telemetry.csv
/content/repo/ppo_vs_sac_comparison.png
/content/repo/README.md
/content/repo/requirements.txt
/content/repo/runs/flight_ppo_v4/acmi_snapshots/snapshot_1000000.acmi
/content/repo/runs/flight_ppo_v4/acmi_snapshots/snapshot_100000.acmi
/content/repo/runs/flight_ppo_v4/acmi_snapshots/snapshot_10000.acmi
/content/repo/runs/flight_ppo_v4/acmi_snapshots/snapshot_110000.acmi
/content/repo/runs/flight_ppo_v4/acmi_snapshots/snapshot_120000.acmi
/content/repo/runs/flight_ppo_v4/acmi_snapshots/snapshot_130000.acmi
/content/repo/runs/flight_ppo_v4/acmi_snapshots/snapshot_140000.acmi
/content/repo/runs/f

In [8]:
%%writefile /content/repo/src/drone_rl/utils/units.py
"""Birim donusumleri. JSBSim emperyal birim kullanir."""

FT2M = 0.3048
M2FT = 1.0 / FT2M

def ft_to_m(x):
    return x * FT2M

def m_to_ft(x):
    return x * M2FT

Overwriting /content/repo/src/drone_rl/utils/units.py


In [9]:
%%writefile /content/repo/src/drone_rl/policies.py

"""F450FlightEnv icin custom policy bilesenleri.

F450FlightEnv'in 15 boyutlu gozlemi (bkz. f450_flight_env.py _get_obs()):

    idx  0    : alt_err        (irtifa hatasi)
    idx  1    : hdot           (dikey hiz)
    idx  2    : along_n        (ileri hiz, hedef yone gore)
    idx  3    : cross_n        (yana kayma, hedef yone gore)
    idx  4    : roll
    idx  5    : pitch
    idx  6-8  : p, q, r        (acisal hizlar)
    idx  9-10 : sin(heading), cos(heading)
    idx 11-14 : onceki aksiyon (4 motor)

Varsayilan MlpPolicy bu 15 sayiyi tek bir duz katmana verir; ag hangi
sayinin neye karsilik geldigini sifirdan kesfetmek zorunda kalir.

FlightFeaturesExtractor bunun yerine gozlemi 4 fiziksel gruba ayirip
HER GRUBA AYRI kucuk bir MLP uygular, sonuclari birlestirip tek bir
ozellik vektorune indirger. Boylece agin "dikey durum" ve "yatay/yon
durumu" icin ayri, temiz bir ic temsili olur - bu da f450_flight_env.py
'deki progress-agirlikli odul seklini (once dikey, sonra yatay) ogrenmeyi
kolaylastirmasi beklenir.

Bu extractor hem actor (pi) hem critic (vf) tarafindan PAYLASILIR
(SB3'un varsayilan mimarisi boyle calisir); ayri pi/vf agirliklari
ise zaten train.py'deki net_arch=dict(pi=.., vf=..) ile saglaniyor.
"""

import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from gymnasium import spaces


# F450FlightEnv._get_obs() ile BIREBIR ayni sirada olmali.
VERTICAL_IDX = [0, 1]           # alt_err, hdot
HORIZONTAL_IDX = [2, 3, 9, 10]  # along_n, cross_n, sin_h, cos_h
ATTITUDE_IDX = [4, 5, 6, 7, 8]  # roll, pitch, p, q, r
ACTION_IDX = [11, 12, 13, 14]   # onceki aksiyon (4 motor)


def _branch(in_dim: int, hidden: int, out_dim: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_dim, hidden),
        nn.Tanh(),
        nn.Linear(hidden, out_dim),
        nn.Tanh(),
    )


class FlightFeaturesExtractor(BaseFeaturesExtractor):
    """F450FlightEnv'e ozel, gruplandirilmis-dal (grouped-branch) feature extractor.

    Parametreler
    ----------
    observation_space : gymnasium.spaces.Box, shape (15,) bekleniyor.
    features_dim : birlesik ciktinin boyutu (bundan sonra SB3'un kendi
        net_arch=[128,128] pi/vf katmanlarina girer).
    """

    def __init__(self, observation_space: spaces.Box, features_dim: int = 64):
        super().__init__(observation_space, features_dim)

        obs_dim = observation_space.shape[0]
        if obs_dim != 15:
            raise ValueError(
                f"FlightFeaturesExtractor 15 boyutlu flight gozlemi bekliyor, "
                f"{obs_dim} geldi. (Bu extractor'i hover gorevinde kullanma.)"
            )

        # Indeksleri buffer olarak sakla ki .to(device) ile birlikte tasinsin.
        self.register_buffer("vertical_idx", torch.tensor(VERTICAL_IDX, dtype=torch.long))
        self.register_buffer("horizontal_idx", torch.tensor(HORIZONTAL_IDX, dtype=torch.long))
        self.register_buffer("attitude_idx", torch.tensor(ATTITUDE_IDX, dtype=torch.long))
        self.register_buffer("action_idx", torch.tensor(ACTION_IDX, dtype=torch.long))

        self.vertical_branch = _branch(len(VERTICAL_IDX), 32, 16)
        self.horizontal_branch = _branch(len(HORIZONTAL_IDX), 32, 16)
        self.attitude_branch = _branch(len(ATTITUDE_IDX), 32, 16)
        self.action_branch = _branch(len(ACTION_IDX), 16, 8)

        combined_dim = 16 + 16 + 16 + 8  # = 56
        self.combine = nn.Sequential(
            nn.Linear(combined_dim, features_dim),
            nn.Tanh(),
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        vertical = self.vertical_branch(observations.index_select(1, self.vertical_idx))
        horizontal = self.horizontal_branch(observations.index_select(1, self.horizontal_idx))
        attitude = self.attitude_branch(observations.index_select(1, self.attitude_idx))
        prev_action = self.action_branch(observations.index_select(1, self.action_idx))

        combined = torch.cat([vertical, horizontal, attitude, prev_action], dim=1)
        return self.combine(combined)



Overwriting /content/repo/src/drone_rl/policies.py


In [10]:
%%writefile /content/repo/src/drone_rl/envs/f450_flight_env.py
"""F450 quadcopter icin hedef irtifa + hedef yon (heading) takip gorevi.

f450_env.py'deki F450HoverEnv'e HIC dokunulmadan, ayri bir env olarak
eklenmistir. Heading burada 'hareket yonu' (course over ground) anlamina
gelir, burun yonu (yaw) degil - yani drone burnu farkli yone bakarken bile
komut edilen yonde ilerleyebilir.

DUZELTME (v2):
- Drone artik hedef irtifanin ALTINDAN spawn oluyor (once bir tirmanma
  gerceklesiyor), hedefin hemen yaninda degil.
- Hedef irtifaya ulasip orada bir sure kalinca episode BASARIYLA
  sonlaniyor (terminated=True + success_bonus), sadece sure dolunca
  degil.
- Heading/hiz odulu, irtifa ilerlemesiyle orantili agirliklandiriliyor:
  drone hala tirmanirken yon odulu zayif, hedefe yaklastikca guclenir.
  Boylece 'once yuksel, sonra yavasca hedef yone don' seklinde bir
  yay/spiral davranisi ortaya cikmasi tesvik edilir.
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import jsbsim


class F450FlightEnv(gym.Env):
    """JSBSim F450 modeli uzerinde: hedef irtifaya TIRMAN, tirmanirken
    yavasca hedef yone (dunya cercevesinde, pusula konvansiyonu:
    0=Kuzey, 90=Dogu) don, hedef irtifaya ulasinca episode'u basariyla
    bitir (reset) gorevi.

    Her episode'da target_altitude VE target_heading RASTGELE secilir
    (F450HoverEnv'de target_altitude sabitti). Model bu ikisini gozlem
    olarak alir (alt_err + sin/cos(heading)), yani 'goal-conditioned' bir
    politika ogrenir.
    """

    metadata = {"render_modes": []}

    def __init__(
        self,
        target_altitude_min_ft=20.0,
        target_altitude_max_ft=45.0,
        target_speed_fps=6.0,
        episode_seconds=60.0,
        physics_hz=240,
        control_hz=20,
        hover_throttle=0.420,
        throttle_range=0.25,
        reward_alt_weight=0.10,
        reward_heading_weight=0.08,
        reward_tilt_weight=0.05,
        reward_spin_weight=0.10,
        reward_jerk_weight=0.05,
        crash_penalty=50.0,
        crash_min_alt_ft=1.0,
        crash_max_alt_offset_ft=60.0,
        crash_max_tilt_rad=1.0,
        # --- YENI parametreler ---
        altitude_start_offset_ft=25.0,
        altitude_start_jitter_ft=2.0,
        success_alt_tol_ft=1.5,
        success_hold_seconds=1.0,
        success_bonus=20.0,
    ):
        super().__init__()

        physics_hz = int(physics_hz)
        control_hz = int(control_hz)
        if physics_hz <= 0 or control_hz <= 0:
            raise ValueError("physics_hz ve control_hz pozitif olmali")
        if physics_hz % control_hz != 0:
            raise ValueError(
                f"physics_hz ({physics_hz}) control_hz'e ({control_hz}) tam "
                "bolunmeli. Or: 240/20=12 OK, 240/50 HATALI."
            )

        # Gozlem: alt_err, hdot, along_track, cross_track, roll, pitch,
        #         p, q, r, sin(heading), cos(heading), prev_action(4) = 15
        self.action_space = spaces.Box(-1.0, 1.0, shape=(4,), dtype=np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(15,), dtype=np.float32)

        self.target_altitude_min_ft = target_altitude_min_ft
        self.target_altitude_max_ft = target_altitude_max_ft
        self.target_speed_fps = target_speed_fps

        self.physics_hz = physics_hz
        self.physics_dt = 1.0 / physics_hz
        self.control_hz = control_hz
        self.substeps = physics_hz // control_hz
        self.max_steps = int(episode_seconds * control_hz)

        self.hover_throttle = hover_throttle
        self.throttle_range = throttle_range

        self.reward_alt_weight = reward_alt_weight
        self.reward_heading_weight = reward_heading_weight
        self.reward_tilt_weight = reward_tilt_weight
        self.reward_spin_weight = reward_spin_weight
        self.reward_jerk_weight = reward_jerk_weight
        self.crash_penalty = crash_penalty
        self.crash_min_alt_ft = crash_min_alt_ft
        self.crash_max_alt_offset_ft = crash_max_alt_offset_ft
        self.crash_max_tilt_rad = crash_max_tilt_rad

        # --- YENI: baslangic offseti ve basari (success) ayarlari ---
        self.altitude_start_offset_ft = altitude_start_offset_ft
        self.altitude_start_jitter_ft = altitude_start_jitter_ft
        self.success_alt_tol_ft = success_alt_tol_ft
        self.success_hold_steps = max(1, int(success_hold_seconds * control_hz))
        self.success_bonus = success_bonus
        self._success_counter = 0
        # Ilerleme (progress) hesaplamak icin: episode basindaki irtifa farki
        self._initial_alt_err_ft = 1.0  # reset()'te gercek degerle guncellenir

        # Her episode'da rastgele secilecek hedefler - reset()'te doldurulur
        self.target_altitude = (target_altitude_min_ft + target_altitude_max_ft) / 2.0
        self.target_heading = 0.0

        self.fdm = jsbsim.FGFDMExec(None)
        self.fdm.set_debug_level(0)
        if not self.fdm.load_model("F450"):
            raise RuntimeError("F450 modeli yuklenemedi")
        self.fdm.set_dt(self.physics_dt)

        self.step_count = 0
        self.prev_action = np.zeros(4, dtype=np.float32)

    @property
    def control_dt(self):
        return self.substeps * self.physics_dt

    def _apply_initial_conditions(self):
        # DUZELTME: drone artik hedef irtifanin ALTINDAN basliyor, hemen
        # yaninda degil - boylece gercek bir tirmanma gerceklesir.
        jitter = self.np_random.uniform(
            -self.altitude_start_jitter_ft, self.altitude_start_jitter_ft
        )
        h0 = self.target_altitude - self.altitude_start_offset_ft + jitter
        # Yerden/crash siniri altina dusmesin diye guvenlik payi birak.
        h0 = max(h0, self.crash_min_alt_ft + 3.0)
        self.fdm["ic/h-agl-ft"] = h0

        self.fdm["ic/u-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/v-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/w-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/phi-rad"] = self.np_random.uniform(-0.05, 0.05)
        self.fdm["ic/theta-rad"] = self.np_random.uniform(-0.05, 0.05)
        self.fdm["ic/psi-true-rad"] = 0.0

        return abs(h0 - self.target_altitude)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # her episode'da hedef irtifa VE hedef yon rastgele secilir
        self.target_altitude = self.np_random.uniform(
            self.target_altitude_min_ft, self.target_altitude_max_ft
        )
        self.target_heading = self.np_random.uniform(0.0, 2.0 * np.pi)

        initial_alt_err = self._apply_initial_conditions()
        self._initial_alt_err_ft = max(initial_alt_err, 1e-3)
        self.fdm.run_ic()

        for i in range(4):
            self.fdm[f"propulsion/engine[{i}]/set-running"] = 1
        self.fdm["fcs/ScasEngage"] = 0

        for i in range(4):
            self.fdm[f"fcs/throttle-cmd-norm[{i}]"] = self.hover_throttle

        self.step_count = 0
        self.prev_action = np.zeros(4, dtype=np.float32)
        self._success_counter = 0

        return self._get_obs(), {}

    def _world_frame_velocity(self):
        """Govde-cercevesi (body-frame) u,v hizlarini, mevcut yaw (psi)
        acisini kullanarak dunya-cercevesi (kuzey, dogu) hizlarina cevirir.
        Yaw zamanla suruklenirse bile bu donusum otomatik dogru kalir."""
        f = self.fdm
        u = f["velocities/u-fps"]
        v = f["velocities/v-fps"]
        psi = f["attitude/psi-rad"]
        north_vel = u * np.cos(psi) - v * np.sin(psi)
        east_vel = u * np.sin(psi) + v * np.cos(psi)
        return north_vel, east_vel

    def _along_cross_track(self):
        """Dunya-cercevesi hizi, hedef yone gore 'ileri (along-track)' ve
        'yana kayma (cross-track)' bilesenlerine ayirir. Aci hesabi
        (atan2) kullanmadigimiz icin dusuk hizda tekillik/gurultu olmaz."""
        north_vel, east_vel = self._world_frame_velocity()
        h = self.target_heading
        along = north_vel * np.cos(h) + east_vel * np.sin(h)
        cross = -north_vel * np.sin(h) + east_vel * np.cos(h)
        return along, cross

    def _climb_progress(self, alt_err_ft):
        """0 (hala baslangic irtifasinda) -> 1 (hedef irtifaya ulasti)
        arasinda bir ilerleme skoru. Heading odulunu bununla carparak
        drone once yukselirken yon odulunu zayif tutuyoruz, hedefe
        yaklastikca guclendiriyoruz -> 'yay/spiral' davranisi."""
        progress = 1.0 - (alt_err_ft / self._initial_alt_err_ft)
        return float(np.clip(progress, 0.0, 1.0))

    def _get_obs(self):
        f = self.fdm
        alt_err = (f["position/h-agl-ft"] - self.target_altitude) / 10.0
        hdot = f["velocities/h-dot-fps"] / 10.0

        along, cross = self._along_cross_track()
        along_n = along / 10.0
        cross_n = cross / 10.0

        roll = f["attitude/phi-rad"]
        pitch = f["attitude/theta-rad"]
        p = f["velocities/p-rad_sec"] / 5.0
        q = f["velocities/q-rad_sec"] / 5.0
        r = f["velocities/r-rad_sec"] / 5.0

        sin_h = np.sin(self.target_heading)
        cos_h = np.cos(self.target_heading)

        return np.array(
            [alt_err, hdot, along_n, cross_n, roll, pitch, p, q, r,
             sin_h, cos_h, *self.prev_action],
            dtype=np.float32,
        )

    def _get_telemetry(self, crashed, reached):
        f = self.fdm
        alt_agl_ft = float(f["position/h-agl-ft"])
        along, cross = self._along_cross_track()
        return {
            "alt_ft": alt_agl_ft,
            "alt_sl_ft": float(f["position/h-sl-ft"]),
            "alt_err_ft": abs(alt_agl_ft - self.target_altitude),
            "lat_deg": float(f["position/lat-geod-deg"]),
            "lon_deg": float(f["position/long-gc-deg"]),
            "x_m": float(f["position/distance-from-start-lon-mt"]),
            "y_m": float(f["position/distance-from-start-lat-mt"]),
            "roll_rad": float(f["attitude/phi-rad"]),
            "pitch_rad": float(f["attitude/theta-rad"]),
            "yaw_rad": float(f["attitude/psi-rad"]),
            "target_heading_rad": float(self.target_heading),
            "along_track_fps": float(along),
            "cross_track_fps": float(cross),
            "target_speed_fps": float(self.target_speed_fps),
            "crashed": bool(crashed),
            "reached_target": bool(reached),
        }

    def _is_crashed(self):
        alt = self.fdm["position/h-agl-ft"]
        return (
            alt < self.crash_min_alt_ft
            or alt > self.target_altitude + self.crash_max_alt_offset_ft
            or abs(self.fdm["attitude/phi-rad"]) > self.crash_max_tilt_rad
            or abs(self.fdm["attitude/theta-rad"]) > self.crash_max_tilt_rad
        )

    def step(self, action):
        action = np.asarray(action, dtype=np.float32).reshape(4)

        throttles = np.clip(
            self.hover_throttle + action * self.throttle_range, 0.0, 1.0
        )

        for _ in range(self.substeps):
            for i in range(4):
                self.fdm[f"fcs/throttle-cmd-norm[{i}]"] = float(throttles[i])
            self.fdm.run()

        self.step_count += 1
        obs = self._get_obs()

        alt_err_ft = abs(self.fdm["position/h-agl-ft"] - self.target_altitude)
        along, cross = self._along_cross_track()

        # DUZELTME: heading/hiz odulu, tirmanma ilerlemesiyle carpiliyor.
        # Drone hala baslangic irtifasindaysa (progress~0) yon odulu
        # zayif; hedefe yaklastikca (progress->1) guclenir. Boylece
        # ajan once dikey harekete, sonra yatay harekete agirlik verir.
        progress = self._climb_progress(alt_err_ft)
        heading_err = abs(self.target_speed_fps - along) + abs(cross)

        tilt = abs(self.fdm["attitude/phi-rad"]) + abs(self.fdm["attitude/theta-rad"])
        spin = abs(self.fdm["velocities/p-rad_sec"]) + abs(self.fdm["velocities/q-rad_sec"])
        jerk = float(np.sum(np.abs(action - self.prev_action)))

        reward = (
            1.0
            - self.reward_alt_weight * alt_err_ft
            - self.reward_heading_weight * progress * heading_err
            - self.reward_tilt_weight * tilt
            - self.reward_spin_weight * spin
            - self.reward_jerk_weight * jerk
        )

        crashed = self._is_crashed()
        if crashed:
            reward -= self.crash_penalty

        # DUZELTME: hedef irtifaya ulasip bir sure orada kalinca
        # (success_hold_steps) episode BASARIYLA sonlanir.
        if alt_err_ft < self.success_alt_tol_ft:
            self._success_counter += 1
        else:
            self._success_counter = 0

        reached = self._success_counter >= self.success_hold_steps
        if reached:
            reward += self.success_bonus

        info = self._get_telemetry(crashed, reached)

        self.prev_action = action.copy()

        terminated = bool(crashed or reached)
        truncated = bool(self.step_count >= self.max_steps)

        return obs, float(reward), terminated, truncated, info



Overwriting /content/repo/src/drone_rl/envs/f450_flight_env.py


In [12]:
%%writefile /content/repo/src/drone_rl/visualize.py
"""Egitilmis PPO politikasini calistirip 3D animasyon icin telemetri toplar."""

import json
import numpy as np
from stable_baselines3.common.vec_env import VecNormalize

from drone_rl.config import load_config
from drone_rl.env_factory import make_eval_vec_env, make_flight_eval_vec_env
from drone_rl.evaluate import resolve_model_paths, ALGO_CLASSES


def export_telemetry_json(telem, path):
    serializable = {}
    for k, v in telem.items():
        if hasattr(v, "tolist"):
            serializable[k] = v.tolist()
        else:
            serializable[k] = v
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f)
    print(f"Telemetri JSON kaydedildi: {path}")


def run_inference_episode(algo, run, config=None, task="hover", use_best=False, deterministic=True):
    """task='hover' (varsayilan, eski davranis) veya task='flight'."""
    cfg = load_config(config)
    from pathlib import Path
    run = Path(run)
    model_path, vecnorm_path = resolve_model_paths(run, use_best)

    if not vecnorm_path.exists():
        raise FileNotFoundError(f"VecNormalize dosyasi bulunamadi: {vecnorm_path}")

    if task == "hover":
        venv = make_eval_vec_env(cfg.env)
    else:
        venv = make_flight_eval_vec_env(cfg.flight_env)

    venv = VecNormalize.load(str(vecnorm_path), venv)
    venv.training = False
    venv.norm_reward = False

    model = ALGO_CLASSES[algo].load(str(model_path), device="cpu")
    raw = venv.envs[0]
    control_dt = raw.control_dt

    records = []
    obs = venv.reset()
    t = 0.0

    while True:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, _, done, infos = venv.step(action)
        telem = infos[0]

        # Aksiyon (-1..+1) yerine gercek motor gucunu (0..1) kaydediyoruz -
        # env'in kendi throttle donusum formulu ile ayni.
        motor_throttle = np.clip(
            raw.hover_throttle + action[0] * raw.throttle_range, 0.0, 1.0
        )

        rec = {
            "t": t,
            "x_m": telem["x_m"],
            "y_m": telem["y_m"],
            "alt_ft": telem["alt_ft"],
            "roll_rad": telem["roll_rad"],
            "pitch_rad": telem["pitch_rad"],
            "yaw_rad": telem["yaw_rad"],
            "alt_err_ft": telem["alt_err_ft"],
            "crashed": telem["crashed"],
            "motor_throttle": motor_throttle.tolist(),
        }
        if "target_heading_rad" in telem:
            rec["target_heading_rad"] = telem["target_heading_rad"]
            rec["along_track_fps"] = telem["along_track_fps"]
            rec["cross_track_fps"] = telem["cross_track_fps"]
        records.append(rec)
        t += control_dt
        if done[0]:
            break

    out = {k: np.array([r[k] for r in records]) for k in records[0].keys()}
    out["control_dt"] = control_dt
    out["target_altitude_ft"] = raw.target_altitude
    return out

Overwriting /content/repo/src/drone_rl/visualize.py


In [ ]:
!ls -la /content/runs/

ls: cannot access '/content/runs/': No such file or directory


In [13]:
%%writefile /content/repo/src/drone_rl/train.py
"""F450 ucus gorevleri icin PPO egitimi + EvalCallback.

--task hover  -> F450HoverEnv (sabit irtifa)
--task flight -> F450FlightEnv (rastgele irtifa + rastgele heading)
"""

import argparse
from pathlib import Path
import numpy as np
import torch.nn as nn
from drone_rl.acmi_writer import ACMIWriter
from drone_rl.utils.units import ft_to_m

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback

from drone_rl.config import load_config
from drone_rl.env_factory import (
    make_training_vec_env, make_flight_training_vec_env,
)
from drone_rl.policies import FlightFeaturesExtractor


class SaveVecNormalizeCallback(BaseCallback):
    def __init__(self, save_path: Path):
        super().__init__()
        self.save_path = Path(save_path)

    def _on_step(self) -> bool:
        vec_normalize = self.model.get_vec_normalize_env()
        if vec_normalize is not None:
            self.save_path.mkdir(parents=True, exist_ok=True)
            vec_normalize.save(str(self.save_path / "vecnormalize_best.pkl"))
        return True


class ACMISnapshotCallback(BaseCallback):
    def __init__(self, eval_env, out_dir: Path, eval_freq: int, control_dt: float):
        super().__init__()
        self.eval_env = eval_env
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.eval_freq = eval_freq
        self.control_dt = control_dt

    def _on_step(self) -> bool:
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            self._record_snapshot()
        return True

    def _record_snapshot(self):
        acmi = ACMIWriter(name="F450", obj_type="Air+Rotorcraft+UAV", color="Blue")
        obs = self.eval_env.reset()
        t = 0.0

        while True:
            action, _ = self.model.predict(obs, deterministic=True)
            obs, _, done, infos = self.eval_env.step(action)
            telem = infos[0]

            acmi.add_frame(
                t=t,
                lon_deg=telem["lon_deg"],
                lat_deg=telem["lat_deg"],
                alt_m=ft_to_m(telem["alt_sl_ft"]),
                roll_deg=np.degrees(telem["roll_rad"]),
                pitch_deg=np.degrees(telem["pitch_rad"]),
                yaw_deg=np.degrees(telem["yaw_rad"]),
            )
            t += self.control_dt

            if done[0]:
                break

        path = self.out_dir / f"snapshot_{self.num_timesteps}.acmi"
        acmi.save(path)
        print(f"  [ACMI] snapshot kaydedildi: {path}", flush=True)


ACTIVATION_MAP = {"tanh": nn.Tanh, "relu": nn.ReLU}


def build_policy_kwargs(cfg_ppo, task: str):
    """policy_kwargs'i olustur: her zaman net_arch/activation'i uygular;
    task='flight' VE cfg_ppo.use_custom_extractor=True ise ayrica
    FlightFeaturesExtractor'i (gruplandirilmis-dal ozellik cikarici)
    de ekler. Hover gorevinde bu extractor hic kullanilmiyor - hover'in
    gozlem boyutu/duzeni farkli (13-dim), extractor 15-dim bekliyor."""
    kwargs = {}

    pi_arch = cfg_ppo.net_arch_pi if cfg_ppo.net_arch_pi is not None else [64, 64]
    vf_arch = cfg_ppo.net_arch_vf if cfg_ppo.net_arch_vf is not None else [64, 64]
    kwargs["net_arch"] = dict(pi=pi_arch, vf=vf_arch)

    if cfg_ppo.activation_fn is not None:
        act_key = cfg_ppo.activation_fn.lower()
        if act_key not in ACTIVATION_MAP:
            raise ValueError(f"Bilinmeyen activation_fn: {cfg_ppo.activation_fn!r}")
        kwargs["activation_fn"] = ACTIVATION_MAP[act_key]

    if task == "flight" and getattr(cfg_ppo, "use_custom_extractor", False):
        kwargs["features_extractor_class"] = FlightFeaturesExtractor
        kwargs["features_extractor_kwargs"] = dict(
            features_dim=getattr(cfg_ppo, "features_dim", 64)
        )

    return kwargs


def build_model(algo: str, cfg, venv, tensorboard_log: str, task: str):
    if algo != "ppo":
        raise ValueError(f"Bilinmeyen algoritma: {algo!r} (sadece ppo destekleniyor)")

    policy_kwargs = build_policy_kwargs(cfg.ppo, task)
    return PPO(
        cfg.ppo.policy, venv,
        n_steps=cfg.ppo.n_steps, batch_size=cfg.ppo.batch_size, n_epochs=cfg.ppo.n_epochs,
        gamma=cfg.ppo.gamma, gae_lambda=cfg.ppo.gae_lambda, clip_range=cfg.ppo.clip_range,
        learning_rate=cfg.ppo.learning_rate, ent_coef=cfg.ppo.ent_coef,
        policy_kwargs=policy_kwargs,
        verbose=1, device="cpu",
        tensorboard_log=tensorboard_log,
    )


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--algo", type=str, choices=["ppo"], default="ppo")
    ap.add_argument("--task", type=str, choices=["hover", "flight"], default="hover",
                     help="hover: sabit irtifa | flight: rastgele irtifa+heading")
    ap.add_argument("--config", type=str, default=None)
    ap.add_argument("--timesteps", type=int, default=None)
    ap.add_argument("--n-envs", type=int, default=None)
    ap.add_argument("--out", type=str, default="/content/runs/run")
    ap.add_argument("--eval-freq", type=int, default=10000)
    args = ap.parse_args()

    cfg = load_config(args.config)
    timesteps = args.timesteps if args.timesteps is not None else cfg.train.timesteps
    n_envs = args.n_envs if args.n_envs is not None else cfg.train.n_envs

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    if args.task == "hover":
        venv = make_training_vec_env(cfg.env, n_envs=n_envs, training=True, norm_reward=True)
        eval_env = make_training_vec_env(cfg.env, n_envs=1, training=False, norm_reward=False)
    else:
        venv = make_flight_training_vec_env(cfg.flight_env, n_envs=n_envs, training=True, norm_reward=True)
        eval_env = make_flight_training_vec_env(cfg.flight_env, n_envs=1, training=False, norm_reward=False)

    model = build_model(args.algo, cfg, venv, tensorboard_log=str(out / "tb"), task=args.task)

    ckpt_cb = CheckpointCallback(
        save_freq=max(20_000 // n_envs, 1),
        save_path=str(out / "ckpt"),
        name_prefix=args.algo,
    )

    best_model_path = out / "best_model"
    save_vecnorm_cb = SaveVecNormalizeCallback(best_model_path)

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=str(best_model_path),
        callback_on_new_best=save_vecnorm_cb,
        log_path=str(out / "logs"),
        eval_freq=max(args.eval_freq // n_envs, 1),
        deterministic=True,
        render=False,
    )

    raw_eval_env = eval_env.venv.envs[0].unwrapped
    acmi_snapshot_cb = ACMISnapshotCallback(
        eval_env=eval_env,
        out_dir=out / "acmi_snapshots",
        eval_freq=max(args.eval_freq // n_envs, 1),
        control_dt=raw_eval_env.control_dt,
    )

    model.learn(total_timesteps=timesteps, callback=[ckpt_cb, eval_cb, acmi_snapshot_cb])

    model.save(out / "model_final")
    venv.save(str(out / "vecnormalize.pkl"))
    print(f"Egitim tamamlandi ve kaydedildi ({args.algo}, task={args.task}):", out)
    print("  Son model      :", out / "model_final.zip", "+", out / "vecnormalize.pkl")
    print("  En iyi model   :", best_model_path / "best_model.zip", "+", best_model_path / "vecnormalize_best.pkl")


if __name__ == "__main__":
    main()





Overwriting /content/repo/src/drone_rl/train.py


In [ ]:
#kayıtlı telemetryden json oluşturucu
from drone_rl.visualize import run_inference_episode, export_telemetry_json

telem = run_inference_episode(
    algo="ppo",
    run="/content/repo/runs/flight_ppo_5sep",
    config="/content/repo/configs/ppo_flight.yaml",
    task="flight",
)
export_telemetry_json(telem, "/content/repo/telemetry_for_html.json")

from google.colab import files
files.download('/content/repo/telemetry_for_html.json')


Telemetri JSON kaydedildi: /content/repo/telemetry_for_html.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#eğer eval.py den dolayı motor verileri gelmezse çalıştır tekrar telemetry ve json oluştur.
import importlib
import drone_rl.config, drone_rl.env_factory, drone_rl.evaluate, drone_rl.visualize

importlib.reload(drone_rl.config)
importlib.reload(drone_rl.env_factory)
importlib.reload(drone_rl.evaluate)
importlib.reload(drone_rl.visualize)

from drone_rl.visualize import run_inference_episode, export_telemetry_json

In [25]:
#ppo flight eğitim
!cd /content/repo/src && python -m drone_rl.train --task flight --timesteps 1000000 --config ../configs/ppo_flight.yaml --out /content/repo/runs/flight_ppo_v4

Streaming output truncated to the last 5000 lines.
|    policy_gradient_loss | -0.0616     |
|    std                  | 0.726       |
|    value_loss           | 0.000654    |
-----------------------------------------
  [ACMI] snapshot kaydedildi: /content/repo/runs/flight_ppo_v4/acmi_snapshots/snapshot_220000.acmi
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1.16e+03  |
|    ep_rew_mean     | -1.41e+03 |
| time/              |           |
|    fps             | 388       |
|    iterations      | 54        |
|    time_elapsed    | 569       |
|    total_timesteps | 221184    |
----------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 1.18e+03   |
|    ep_rew_mean          | -1.44e+03  |
| time/                   |            |
|    fps                  | 390        |
|    iterations           | 55         |
|    time_elapsed         | 576        |
|   

In [28]:
#fligth modeli için eval ve acmi kaydedici
!cd /content/repo/src && python -m drone_rl.evaluate \
    --algo ppo --task flight \
    --config ../configs/ppo_flight.yaml \
    --run /content/repo/runs/flight_ppo_v4 \
    --output /content/repo/ppo_flight_telemetry.csv \
    --acmi-output /content/repo/ppo_flight_final.acmi \
    --episodes 5


2026-09-08 06:50:12.444691: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-08 06:50:12.698596: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT.

In [ ]:

#sac eğitim hover için başlat
#!cd /content/repo/src && python -m drone_rl.train --algo sac --config ../configs/ppo_hover.yaml --timesteps 600000 --n-envs 4 --out /content/runs/hover_sac_v1

#ppo ve sac için ayrı ayrı hover evaluation ama last modeli alır
#!cd /content/repo/src && python -m drone_rl.evaluate --algo ppo --config ../configs/ppo_hover.yaml --run /content/runs/hover_ppo_v1 --output /content/ppo_telemetry.csv --acmi-output /content/ppo_final.acmi --episodes 3
#!cd /content/repo/src && python -m drone_rl.evaluate --algo sac --config ../configs/ppo_hover.yaml --run /content/runs/hover_sac_v1 --output /content/sac_telemetry.csv --acmi-output /content/sac_final.acmi --episodes 3

#yine ppo ve sac için hover evaluation kayıtları ama best model kullanır last değil
#!cd /content/repo/src && python -m drone_rl.evaluate --algo ppo --config ../configs/ppo_hover.yaml --run /content/runs/hover_ppo_v1 --output /content/ppo_best_telemetry.csv --episodes 5 --use-best
#!cd /content/repo/src && python -m drone_rl.evaluate --algo sac --config ../configs/ppo_hover.yaml --run /content/runs/hover_sac_v1 --output /content/sac_best_telemetry.csv --episodes 5 --use-best

#hyperparameter optimization via optuna
#!cd /content/repo/src && python -m drone_rl.tune --algo ppo --n-trials 20 --timesteps-per-trial 60000 --out /content/tuning/ppo
#!cd /content/repo/src && python -m drone_rl.tune --algo sac --n-trials 20 --timesteps-per-trial 60000 --out /content/tuning/sac

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT. RESULTS WILL BE UNPREDICTABLE !!
Current version needed is: 2.0
         You have version: 3.0

Failed to tie property fcs/accelx/malfunction/fail_low to object methods
Failed to tie property fcs/accelx/malfunction/fail_high to object methods
Failed to tie property fcs/accelx/malfunction/fail_stuck to object methods
Failed to tie property fcs/accelx/randomseed to object methods
Failed to tie property fcs/accely/malfunction/fail_low to object methods
Faile

In [ ]:
#optuna ile optimize edilen parametreler
!cat /content/tuning/ppo/best_params_ppo.json

{
  "algo": "ppo",
  "best_value": 333.0058974,
  "best_params": {
    "n_steps": 1024,
    "learning_rate": 0.0009682086811397772,
    "batch_size": 256,
    "n_epochs": 8,
    "gamma": 0.98,
    "gae_lambda": 0.821733218323567,
    "clip_range": 0.30028032106674823,
    "ent_coef": 0.008527398284507002
  }
}

In [14]:
%%writefile /content/repo/src/drone_rl/evaluate.py
"""Egitilmis PPO politikasini calistir; CSV telemetri ve/veya
gercek ACMI (Tacview) dosyasi olarak kaydet.

--task hover  -> F450HoverEnv
--task flight -> F450FlightEnv
"""

import argparse
from pathlib import Path
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize

from drone_rl.config import load_config
from drone_rl.env_factory import make_eval_vec_env, make_flight_eval_vec_env
from drone_rl.utils.units import ft_to_m
from drone_rl.acmi_writer import ACMIWriter


ALGO_CLASSES = {"ppo": PPO}


def resolve_model_paths(run: Path, use_best: bool):
    if use_best:
        return run / "best_model" / "best_model", run / "best_model" / "vecnormalize_best.pkl"
    return run / "model_final", run / "vecnormalize.pkl"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--algo", type=str, choices=["ppo"], default="ppo")
    ap.add_argument("--task", type=str, choices=["hover", "flight"], default="hover")
    ap.add_argument("--config", type=str, default=None)
    ap.add_argument("--run", type=str, default="/content/repo/runs/run")
    ap.add_argument("--episodes", type=int, default=3)
    ap.add_argument("--output", type=str, default="/content/telemetry.csv")
    ap.add_argument("--acmi-output", type=str, default=None)
    ap.add_argument("--use-best", action="store_true")
    args = ap.parse_args()

    cfg = load_config(args.config)
    run = Path(args.run)
    model_path, vecnorm_path = resolve_model_paths(run, args.use_best)

    if not vecnorm_path.exists():
        raise FileNotFoundError(f"VecNormalize dosyasi bulunamadi: {vecnorm_path}")

    if args.task == "hover":
        venv = make_eval_vec_env(cfg.env)
    else:
        venv = make_flight_eval_vec_env(cfg.flight_env)

    venv = VecNormalize.load(str(vecnorm_path), venv)
    venv.training = False
    venv.norm_reward = False

    algo_cls = ALGO_CLASSES[args.algo]
    model = algo_cls.load(str(model_path), device="cpu")
    raw = venv.envs[0]
    control_dt = raw.control_dt

    all_telemetry = []
    acmi = ACMIWriter(name="F450", obj_type="Air+Rotorcraft+UAV", color="Blue") \
        if args.acmi_output else None

    global_t = 0.0

    for ep in range(args.episodes):
        obs = venv.reset()
        t = 0.0

        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, done, infos = venv.step(action)
            telem = infos[0]

            # Aksiyon (-1..+1) yerine gercek motor gucunu (0..1) kaydediyoruz -
            # env'in kendi throttle donusum formulu ile ayni.
            motor_throttle = np.clip(
                raw.hover_throttle + action[0] * raw.throttle_range, 0.0, 1.0
            )

            data = {
                "timestamp": round(t, 4),
                "episode": ep,
                "alt_ft": round(telem["alt_ft"], 4),
                "roll_rad": round(telem["roll_rad"], 6),
                "pitch_rad": round(telem["pitch_rad"], 6),
                "yaw_rad": round(telem["yaw_rad"], 6),
                "alt_err_ft": round(telem["alt_err_ft"], 4),
                "crashed": telem["crashed"],
                "motor_1": round(float(motor_throttle[0]), 4),
                "motor_2": round(float(motor_throttle[1]), 4),
                "motor_3": round(float(motor_throttle[2]), 4),
                "motor_4": round(float(motor_throttle[3]), 4),
            }
            if "target_heading_rad" in telem:
                data["target_heading_rad"] = round(telem["target_heading_rad"], 4)
                data["along_track_fps"] = round(telem["along_track_fps"], 4)
                data["cross_track_fps"] = round(telem["cross_track_fps"], 4)
            if "reached_target" in telem:
                data["reached_target"] = telem["reached_target"]
            all_telemetry.append(data)

            if acmi is not None:
                alt_m = ft_to_m(telem["alt_sl_ft"])
                acmi.add_frame(
                    t=global_t,
                    lon_deg=telem["lon_deg"],
                    lat_deg=telem["lat_deg"],
                    alt_m=alt_m,
                    roll_deg=np.degrees(telem["roll_rad"]),
                    pitch_deg=np.degrees(telem["pitch_rad"]),
                    yaw_deg=np.degrees(telem["yaw_rad"]),
                )

            t += control_dt
            global_t += control_dt
            if done[0]:
                break

    df = pd.DataFrame(all_telemetry)
    df.to_csv(args.output, index=False, header=True)
    print(f"CSV telemetri kaydedildi: {args.output}")
    print(df.head())

    if acmi is not None:
        saved_path = acmi.save(args.acmi_output)
        print(f"ACMI (Tacview) dosyasi kaydedildi: {saved_path}")


if __name__ == '__main__':
    main()


Overwriting /content/repo/src/drone_rl/evaluate.py


In [16]:
#gitignore oluşturur ve günceller
with open("/content/repo/.gitignore", "w") as f:
    f.write("__pycache__/\n.ipynb_checkpoints\nruns/*/ckpt/\n")
!cat /content/repo/.gitignore

__pycache__/
.ipynb_checkpoints
runs/*/ckpt/


In [17]:
%%writefile /content/repo/configs/ppo_flight.yaml
flight_env:
  target_altitude_min_ft: 20.0
  target_altitude_max_ft: 45.0
  target_speed_fps: 6.0
  episode_seconds: 60.0
  control_hz: 20
  physics_hz: 240
  hover_throttle: 0.420
  throttle_range: 0.25
  reward_alt_weight: 0.10
  reward_heading_weight: 0.08
  reward_tilt_weight: 0.05
  reward_spin_weight: 0.10
  reward_jerk_weight: 0.05
  # --- tirmanma baslangici + hedefe ulasinca reset (success) ---
  altitude_start_offset_ft: 25.0
  altitude_start_jitter_ft: 2.0
  success_alt_tol_ft: 1.5
  success_hold_seconds: 3.0
  success_bonus: 20.0
ppo:
  policy: MlpPolicy
  n_steps: 1024
  batch_size: 256
  n_epochs: 8
  gamma: 0.98
  gae_lambda: 0.821733218323567
  clip_range: 0.30028032106674823
  learning_rate: 0.0009682086811397772
  ent_coef: 0.008527398284507002
  net_arch_pi: [128, 128]
  net_arch_vf: [128, 128]
  activation_fn: tanh
  # --- custom feature extractor (FlightFeaturesExtractor) ---
  use_custom_extractor: true
  features_dim: 64
train:
  timesteps: 600000
  n_envs: 4

Overwriting /content/repo/configs/ppo_flight.yaml


In [ ]:
readme = """# quadcopter-rl-copilot

JSBSim F450 quadcopter modeli uzerinde PPO ile hover kontrolu.

## Sonuc (hover_v1)

300.000 adim egitim sonrasi, 5 degerlendirme episode'unda:

- Episode uzunlugu: 400/400 (hic dusme yok)
- Hedef irtifadan ortalama sapma: 0.02 - 0.06 ft
- Ortalama egilme: 0.02 - 0.08 rad

## Kurulum (Colab)

    !pip install -q stable-baselines3 gymnasium
    !pip uninstall -y -q jsbsim
    !pip install -q jsbsim==1.2.4

Repoyu klonladiktan sonra src dizinini Python yoluna ekle:

    import os, sys
    sys.path.insert(0, "/content/repo/src")
    os.environ["PYTHONPATH"] = "/content/repo/src"

## Kullanim

Egitim:

    cd src && python -m drone_rl.train --timesteps 300000 --n-envs 4 --out ../runs/hover_v2

Degerlendirme:

    cd src && python -m drone_rl.evaluate --run ../runs/hover_v1 --csv /content/iz.csv

## Yapi

- src/drone_rl/envs/f450_env.py - Gymnasium ortami
- src/drone_rl/train.py - PPO egitimi
- src/drone_rl/evaluate.py - egitilmis politikanin olculmesi
- configs/ppo_hover.yaml - kullanilan ayarlar
- runs/hover_v1/ - egitilmis model ve normalizasyon istatistikleri
- notebooks/quadcopter_rl.ipynb - Colab calisma defteri

## Notlar

- JSBSim emperyal birim kullanir (ft, lbs, fps).
- Hover gazi 0.410 olarak olculdu. Aksiyon bu deger etrafinda +-0.25
  araliginda olceklenir, boylece sifir aksiyon "asili kal" anlamina gelir.
- vecnormalize.pkl model ile birlikte yuklenmelidir, aksi halde politika
  yanlis olcekli gozlem alir ve calismaz.
- F450 XML'i yuklenirken "version 3.0" uyarisi verir; zararsizdir.
"""

with open("/content/repo/README.md", "w") as f:
    f.write(readme)

print(open("/content/repo/README.md").read()[:300])

# quadcopter-rl-copilot

JSBSim F450 quadcopter modeli uzerinde PPO ile hover kontrolu.

## Sonuc (hover_v1)

300.000 adim egitim sonrasi, 5 degerlendirme episode'unda:

- Episode uzunlugu: 400/400 (hic dusme yok)
- Hedef irtifadan ortalama sapma: 0.02 - 0.06 ft
- Ortalama egilme: 0.02 - 0.08 rad

#


In [18]:
%%writefile /content/repo/src/drone_rl/acmi_writer.py
"""Tacview ACMI (.acmi) format yazici - basit tek-obje coklu-episode kaydi.

ACMI format referansi: https://www.tacview.net/documentation/acmi/en/

Bu writer sadece bu proje icin gerekli minimum alt kumeyi destekler:
tek bir hava araci objesi, zaman serisi konum/aci guncellemeleri.
Coklu obje, olay (event) kayitlari, veya ek property'ler (hiz, RPM vb.)
desteklenmiyor - ihtiyac olursa genisletilebilir.

Beklenen birimler (ACMI standardi):
- lon_deg, lat_deg : derece (WGS84)
- alt_m            : metre (deniz seviyesinden veya yerden - tutarli olmasi yeterli)
- roll_deg, pitch_deg, yaw_deg : derece

JSBSim ft ve radyan kullandigi icin cagiran kod bu donusumu
(units.ft_to_m, np.degrees) yapmali; bu siniftan once cagirilmalidir.
"""

from pathlib import Path
from datetime import datetime, timezone


class ACMIWriter:
    def __init__(self, object_id=1, name="F450", obj_type="Air+Rotorcraft+UAV", color="Blue"):
        self.object_id = object_id
        self.name = name
        self.obj_type = obj_type
        self.color = color
        self._lines = []
        self._header_written = False
        self._object_declared = False
        self._last_frame_time = None

    def _write_header(self):
        now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        self._lines.append("FileType=text/acmi/tacview")
        self._lines.append("FileVersion=2.2")
        self._lines.append(f"0,ReferenceTime={now}")
        self._header_written = True

    def add_frame(self, t, lon_deg, lat_deg, alt_m, roll_deg, pitch_deg, yaw_deg):
        """Bir zaman anindaki obje durumunu ekler.

        t: saniye cinsinden, dosya boyunca MONOTONIK ARTAN olmali
           (birden fazla episode varsa t'yi sifirlama, devam ettir).
        """
        if not self._header_written:
            self._write_header()

        # Ayni t icin tekrar frame acmayalim (float hassasiyeti icin yuvarla)
        t_rounded = round(t, 2)
        if self._last_frame_time != t_rounded:
            self._lines.append(f"#{t_rounded:.2f}")
            self._last_frame_time = t_rounded

        obj_line = (
            f"{self.object_id:x},T={lon_deg:.7f}|{lat_deg:.7f}|{alt_m:.2f}|"
            f"{roll_deg:.2f}|{pitch_deg:.2f}|{yaw_deg:.2f}"
        )
        if not self._object_declared:
            obj_line += f",Name={self.name},Type={self.obj_type},Color={self.color}"
            self._object_declared = True

        self._lines.append(obj_line)

    def save(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write("\n".join(self._lines) + "\n")
        return path


Overwriting /content/repo/src/drone_rl/acmi_writer.py


In [19]:
%%writefile /content/repo/src/drone_rl/config.py
"""Merkezi config yukleme. YAML dosyasindaki degerleri dataclass'lara donusturur."""

from dataclasses import dataclass, field
from typing import Optional, List
import yaml


@dataclass
class EnvConfig:
    """F450HoverEnv icin ayarlar (hedef irtifa sabit)."""
    target_altitude_ft: float = 30.0
    episode_seconds: float = 20.0
    control_hz: int = 20
    physics_hz: int = 240
    hover_throttle: float = 0.420
    throttle_range: float = 0.25
    reward_alt_weight: float = 0.10
    reward_tilt_weight: float = 0.50
    reward_spin_weight: float = 0.10
    reward_jerk_weight: float = 0.05
    crash_penalty: float = 50.0
    crash_min_alt_ft: float = 1.0
    crash_max_alt_offset_ft: float = 60.0
    crash_max_tilt_rad: float = 1.0


@dataclass
class FlightEnvConfig:
    """F450FlightEnv icin ayarlar (hedef irtifa + hedef yon, ikisi de
    her episode'da rastgele). EnvConfig'ten BAGIMSIZ, ayri bir dataclass -
    hover config'ini hic etkilemez.

    Drone hedef irtifanin altitude_start_offset_ft kadar ALTINDAN spawn
    oluyor (gercek bir tirmanma yasaniyor) ve hedef irtifaya ulasip
    success_hold_seconds kadar orada kalinca episode basariyla
    (success_bonus ile) sonlaniyor.
    """
    target_altitude_min_ft: float = 20.0
    target_altitude_max_ft: float = 45.0
    target_speed_fps: float = 6.0
    episode_seconds: float = 60.0
    control_hz: int = 20
    physics_hz: int = 240
    hover_throttle: float = 0.420
    throttle_range: float = 0.25
    reward_alt_weight: float = 0.10
    reward_heading_weight: float = 0.08
    reward_tilt_weight: float = 0.05
    reward_spin_weight: float = 0.10
    reward_jerk_weight: float = 0.05
    crash_penalty: float = 50.0
    crash_min_alt_ft: float = 1.0
    crash_max_alt_offset_ft: float = 60.0
    crash_max_tilt_rad: float = 1.0
    altitude_start_offset_ft: float = 25.0
    altitude_start_jitter_ft: float = 2.0
    success_alt_tol_ft: float = 1.5
    success_hold_seconds: float = 1.0
    success_bonus: float = 20.0


@dataclass
class PPOConfig:
    policy: str = "MlpPolicy"
    n_steps: int = 1024
    batch_size: int = 256
    n_epochs: int = 10
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_range: float = 0.2
    learning_rate: float = 3e-4
    ent_coef: float = 0.0
    net_arch_pi: Optional[List[int]] = None
    net_arch_vf: Optional[List[int]] = None
    activation_fn: Optional[str] = None
    # --- YENI: custom feature extractor (sadece task=flight'ta kullanilir) ---
    use_custom_extractor: bool = False
    features_dim: int = 64


@dataclass
class TrainConfig:
    timesteps: int = 300_000
    n_envs: int = 4


@dataclass
class Config:
    env: EnvConfig = field(default_factory=EnvConfig)
    flight_env: FlightEnvConfig = field(default_factory=FlightEnvConfig)
    ppo: PPOConfig = field(default_factory=PPOConfig)
    train: TrainConfig = field(default_factory=TrainConfig)


def load_config(path: Optional[str]) -> Config:
    if path is None:
        return Config()

    with open(path, "r") as f:
        raw = yaml.safe_load(f) or {}

    return Config(
        env=EnvConfig(**raw.get("env", {})),
        flight_env=FlightEnvConfig(**raw.get("flight_env", {})),
        ppo=PPOConfig(**raw.get("ppo", {})),
        train=TrainConfig(**raw.get("train", {})),
    )






Overwriting /content/repo/src/drone_rl/config.py


In [20]:
%%writefile /content/repo/src/drone_rl/tune.py
"""Optuna ile PPO icin hiperparametre optimizasyonu.

Bu, train.py'nin bir 'modu' degil, ayri bir arac: her calisma (trial)
kisa bir egitim yapip sonucu (ortalama reward) Optuna'ya bildiriyor,
Optuna da bir sonraki denemede hangi hiperparametreleri deneyecegini
bu geri bildirime gore seciyor.

Kullanim:
    python -m drone_rl.tune --algo ppo --n-trials 30 --timesteps-per-trial 60000 --out /content/tuning/ppo
"""

import argparse
import json
from pathlib import Path

import optuna
from optuna.pruners import MedianPruner
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback

from drone_rl.config import load_config
from drone_rl.env_factory import make_training_vec_env


class TrialEvalCallback(EvalCallback):
    """Normal EvalCallback gibi periyodik degerlendirme yapar, ama
    her degerlendirme sonucunu Optuna'ya da raporlar VE ekrana bir
    ilerleme satiri yazdirir."""

    def __init__(self, eval_env, trial, total_timesteps, **kwargs):
        super().__init__(eval_env, **kwargs)
        self.trial = trial
        self.total_timesteps = total_timesteps
        self.eval_idx = 0

    def _on_step(self) -> bool:
        continue_training = super()._on_step()
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            self.eval_idx += 1
            print(
                f"  [trial {self.trial.number}] "
                f"{self.num_timesteps}/{self.total_timesteps} adim - "
                f"ortalama reward: {self.last_mean_reward:.2f}",
                flush=True,
            )
            self.trial.report(self.last_mean_reward, self.eval_idx)
            if self.trial.should_prune():
                print(f"  [trial {self.trial.number}] erken kesildi (prune)", flush=True)
                raise optuna.TrialPruned()
        return continue_training


def suggest_ppo_params(trial: optuna.Trial) -> dict:
    """PPO icin arama uzayi. Yaygin/etkili PPO hiperparametreleri."""
    n_steps = trial.suggest_categorical("n_steps", [512, 1024, 2048])
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
        "n_steps": n_steps,
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256]),
        "n_epochs": trial.suggest_int("n_epochs", 3, 30),
        "gamma": trial.suggest_categorical("gamma", [0.9, 0.95, 0.98, 0.99, 0.995, 0.999]),
        "gae_lambda": trial.suggest_float("gae_lambda", 0.8, 1.0),
        "clip_range": trial.suggest_float("clip_range", 0.1, 0.4),
        "ent_coef": trial.suggest_float("ent_coef", 1e-8, 0.1, log=True),
    }


def build_trial_model(algo: str, params: dict, venv):
    if algo != "ppo":
        raise ValueError(f"Bilinmeyen algoritma: {algo!r} (sadece ppo destekleniyor)")
    return PPO(
        "MlpPolicy", venv,
        n_steps=params["n_steps"], batch_size=params["batch_size"],
        n_epochs=params["n_epochs"], gamma=params["gamma"],
        gae_lambda=params["gae_lambda"], clip_range=params["clip_range"],
        learning_rate=params["learning_rate"], ent_coef=params["ent_coef"],
        verbose=0, device="cpu",
    )


def make_objective(algo: str, cfg, args):
    def objective(trial: optuna.Trial) -> float:
        params = suggest_ppo_params(trial)

        print(f"\n=== Trial {trial.number} basladi ===", flush=True)
        print(f"  Parametreler: {params}", flush=True)

        venv = make_training_vec_env(cfg.env, n_envs=args.n_envs, training=True, norm_reward=True)
        eval_env = make_training_vec_env(cfg.env, n_envs=1, training=False, norm_reward=False)

        model = build_trial_model(algo, params, venv)

        eval_cb = TrialEvalCallback(
            eval_env,
            trial=trial,
            total_timesteps=args.timesteps_per_trial,
            eval_freq=max(args.eval_freq // args.n_envs, 1),
            deterministic=True,
            render=False,
            verbose=0,
        )

        try:
            model.learn(total_timesteps=args.timesteps_per_trial, callback=eval_cb)
        except optuna.TrialPruned:
            venv.close()
            eval_env.close()
            raise

        reward = eval_cb.last_mean_reward
        venv.close()
        eval_env.close()

        print(f"=== Trial {trial.number} bitti - sonuc: {reward:.2f} ===", flush=True)

        if reward is None or reward != reward:
            return -1e6
        return float(reward)

    return objective


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--algo", type=str, choices=["ppo"], default="ppo",
                     help="Hangi algoritma icin hiperparametre aranacak (su an sadece ppo)")
    ap.add_argument("--config", type=str, default=None,
                     help="Ortam ayarlari icin bir config dosyasi "
                          "(sadece env: bolumu kullanilir)")
    ap.add_argument("--n-trials", type=int, default=30)
    ap.add_argument("--timesteps-per-trial", type=int, default=60_000)
    ap.add_argument("--n-envs", type=int, default=4)
    ap.add_argument("--eval-freq", type=int, default=10_000)
    ap.add_argument("--out", type=str, default="/content/tuning/result")
    ap.add_argument("--study-name", type=str, default=None)
    ap.add_argument("--storage", type=str, default=None,
                     help="Optuna icin kalici depolama (ornek: sqlite:////content/tuning/study.db).")
    args = ap.parse_args()

    optuna.logging.set_verbosity(optuna.logging.INFO)

    cfg = load_config(args.config)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=1)
    study = optuna.create_study(
        direction="maximize",
        pruner=pruner,
        study_name=args.study_name or f"{args.algo}_flight",
        storage=args.storage,
        load_if_exists=True,
    )

    print(f"Arama basliyor: algo={args.algo}, n_trials={args.n_trials}, "
          f"timesteps_per_trial={args.timesteps_per_trial}", flush=True)

    study.optimize(make_objective(args.algo, cfg, args), n_trials=args.n_trials)

    print("\n=== Arama tamamlandi ===")
    print("En iyi deger (ortalama reward):", study.best_value)
    print("En iyi parametreler:", study.best_params)

    best_path = out / f"best_params_{args.algo}.json"
    with open(best_path, "w") as f:
        json.dump({"algo": args.algo, "best_value": study.best_value,
                    "best_params": study.best_params}, f, indent=2)
    print("Kaydedildi:", best_path)

    trials_csv = out / f"trials_{args.algo}.csv"
    study.trials_dataframe().to_csv(trials_csv, index=False)
    print("Tum denemeler:", trials_csv)


if __name__ == "__main__":
    main()




Overwriting /content/repo/src/drone_rl/tune.py


In [21]:
%%writefile /content/repo/src/drone_rl/env_factory.py
"""Egitim ve degerlendirme icin ortak F450 ortami/VecEnv kurulum yardimcilari.

Hem F450HoverEnv (hover gorevi) hem F450FlightEnv (irtifa+heading gorevi)
icin ayri fonksiyon setleri barindirir. Biri digerini etkilemez.
"""

from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from drone_rl.envs.f450_env import F450HoverEnv
from drone_rl.envs.f450_flight_env import F450FlightEnv
from drone_rl.config import EnvConfig, FlightEnvConfig


# ---------------------------------------------------------------------
# Hover gorevi (degismedi)
# ---------------------------------------------------------------------

def make_env(env_config: EnvConfig) -> F450HoverEnv:
    return F450HoverEnv(
        target_altitude_ft=env_config.target_altitude_ft,
        episode_seconds=env_config.episode_seconds,
        physics_hz=env_config.physics_hz,
        control_hz=env_config.control_hz,
        hover_throttle=env_config.hover_throttle,
        throttle_range=env_config.throttle_range,
        reward_alt_weight=env_config.reward_alt_weight,
        reward_tilt_weight=env_config.reward_tilt_weight,
        reward_spin_weight=env_config.reward_spin_weight,
        reward_jerk_weight=env_config.reward_jerk_weight,
        crash_penalty=env_config.crash_penalty,
        crash_min_alt_ft=env_config.crash_min_alt_ft,
        crash_max_alt_offset_ft=env_config.crash_max_alt_offset_ft,
        crash_max_tilt_rad=env_config.crash_max_tilt_rad,
    )


def make_training_vec_env(env_config: EnvConfig, n_envs: int, training: bool,
                           norm_reward: bool, clip_obs: float = 10.0):
    def _make():
        return Monitor(make_env(env_config))

    venv = DummyVecEnv([_make for _ in range(n_envs)])
    venv = VecNormalize(
        venv, norm_obs=True, norm_reward=norm_reward, clip_obs=clip_obs, training=training
    )
    return venv


def make_eval_vec_env(env_config: EnvConfig):
    return DummyVecEnv([lambda: make_env(env_config)])


# ---------------------------------------------------------------------
# Flight gorevi (hedef irtifa + hedef yon; artik tirmanma + success reset)
# ---------------------------------------------------------------------

def make_flight_env(flight_config: FlightEnvConfig) -> F450FlightEnv:
    return F450FlightEnv(
        target_altitude_min_ft=flight_config.target_altitude_min_ft,
        target_altitude_max_ft=flight_config.target_altitude_max_ft,
        target_speed_fps=flight_config.target_speed_fps,
        episode_seconds=flight_config.episode_seconds,
        physics_hz=flight_config.physics_hz,
        control_hz=flight_config.control_hz,
        hover_throttle=flight_config.hover_throttle,
        throttle_range=flight_config.throttle_range,
        reward_alt_weight=flight_config.reward_alt_weight,
        reward_heading_weight=flight_config.reward_heading_weight,
        reward_tilt_weight=flight_config.reward_tilt_weight,
        reward_spin_weight=flight_config.reward_spin_weight,
        reward_jerk_weight=flight_config.reward_jerk_weight,
        crash_penalty=flight_config.crash_penalty,
        crash_min_alt_ft=flight_config.crash_min_alt_ft,
        crash_max_alt_offset_ft=flight_config.crash_max_alt_offset_ft,
        crash_max_tilt_rad=flight_config.crash_max_tilt_rad,
        # --- YENI ---
        altitude_start_offset_ft=flight_config.altitude_start_offset_ft,
        altitude_start_jitter_ft=flight_config.altitude_start_jitter_ft,
        success_alt_tol_ft=flight_config.success_alt_tol_ft,
        success_hold_seconds=flight_config.success_hold_seconds,
        success_bonus=flight_config.success_bonus,
    )


def make_flight_training_vec_env(flight_config: FlightEnvConfig, n_envs: int,
                                  training: bool, norm_reward: bool, clip_obs: float = 10.0):
    def _make():
        return Monitor(make_flight_env(flight_config))

    venv = DummyVecEnv([_make for _ in range(n_envs)])
    venv = VecNormalize(
        venv, norm_obs=True, norm_reward=norm_reward, clip_obs=clip_obs, training=training
    )
    return venv


def make_flight_eval_vec_env(flight_config: FlightEnvConfig):
    return DummyVecEnv([lambda: make_flight_env(flight_config)])


Overwriting /content/repo/src/drone_rl/env_factory.py


In [ ]:
%%bash
# Repo dizinine geç
cd /content/repo

# Tüm değişiklikleri ekle
git add .

git commit -m "real-time simulation server and html"
git pull origin main --no-edit
git push origin main

[main 8f7fdfc] real-time simulation server and html
 3 files changed, 761 insertions(+), 1 deletion(-)
 create mode 100644 droneSim_realtime.html
 create mode 100644 src/drone_rl/realtime_server.py
Already up to date.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
From https://github.com/miray7yuce/quadcopter-rl-copilot
 * branch            main       -> FETCH_HEAD
To https://github.com/miray7yuce/quadcopter-rl-copilot.git
   2c4427e..8f7fdfc  main -> main


In [ ]:
from google.colab import files
files.download('/content/ppo_telemetry.csv')
files.download('/content/ppo_final.acmi')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>